# V3: DJF Running Correlation (Rainfall vs Nino3.4)

This notebook only computes and plots the **running correlation** between:
- DJF regional rainfall (MSWEP area-mean)
- DJF Nino3.4 index

for each region in `domain.json`.

DJF definition is strict: **Dec(y-1), Jan(y), Feb(y)**.


## Setup

Load libraries, define paths, and create `output_v3` folders.


In [ ]:
from __future__ import annotations

import json
import os
import re
from collections import OrderedDict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mplconfig"))

BASE_DIR = Path("/Users/rizzie/Academic/9_TugasAkhir")
DATA_MSWEP_DIR = BASE_DIR / "data/mswep"
DATA_NINO34 = BASE_DIR / "data/index/nino34.anom.csv"
DATA_NINA34 = BASE_DIR / "data/index/nina34.anom.csv"
DOMAIN_JSON = BASE_DIR / "data/all_data/domain.json"
NOTEBOOK_DIR = BASE_DIR / "notebook/non-stationarity"

OUT_ROOT = NOTEBOOK_DIR / "output_v3"
OUT_PNG = OUT_ROOT / "png"
OUT_TABLES = OUT_ROOT / "tables"
for d in (OUT_ROOT, OUT_PNG, OUT_TABLES):
    d.mkdir(parents=True, exist_ok=True)


def slugify(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")


def save_and_show(fig: plt.Figure, filename: str, dpi: int = 180) -> Path:
    out_path = OUT_PNG / filename
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print(f"Saved: {out_path}")
    return out_path


## Helper Functions

Helper functions for domain loading, DJF aggregation, Nino3.4 parsing, and running correlation.


In [ ]:
def _iter_xy(coords):
    if isinstance(coords, (list, tuple)) and coords:
        first = coords[0]
        if isinstance(first, (int, float)) and len(coords) >= 2:
            yield float(coords[0]), float(coords[1])
        else:
            for part in coords:
                yield from _iter_xy(part)


def load_domains(path: Path) -> OrderedDict[str, dict]:
    obj = json.loads(path.read_text(encoding="utf-8"))
    feats = obj.get("features", [])
    if not feats:
        raise ValueError(f"No features in {path}")

    domains = OrderedDict()
    used_names = set()

    for i, feat in enumerate(feats, start=1):
        props = feat.get("properties") or {}
        xy = list(_iter_xy(feat.get("geometry", {}).get("coordinates", [])))
        if not xy:
            raise ValueError(f"Feature {i} has empty geometry")

        xs = np.array([p[0] for p in xy], dtype=float)
        ys = np.array([p[1] for p in xy], dtype=float)

        lon_min = float(np.min(xs) % 360)
        lon_max = float(np.max(xs) % 360)
        lat_min = float(np.min(ys))
        lat_max = float(np.max(ys))
        lon_c = 0.5 * (lon_min + lon_max)
        lat_c = 0.5 * (lat_min + lat_max)

        configured_name = props.get("name") or props.get("id") or props.get("domain")
        if configured_name:
            name = str(configured_name)
        else:
            if lon_c < 100 and lat_c > 2:
                name = "Aceh"
            elif lon_c < 106:
                name = "Riau"
            elif 106 <= lon_c < 110 and lat_c < -4:
                name = "West Java"
            elif 110 <= lon_c < 116 and lat_c < -4:
                name = "East Java"
            elif 108 <= lon_c < 116 and lat_c >= -4:
                name = "Kalimantan"
            elif 118 <= lon_c < 125:
                name = "Central Sulawesi"
            elif 130 <= lon_c < 136:
                name = "West Papua"
            elif 136 <= lon_c <= 142 and lat_c > -4.5:
                name = "North Papua"
            elif 136 <= lon_c <= 142 and lat_c <= -4.5:
                name = "South Papua"
            else:
                name = f"domain_{i:02d}"

        if name in used_names:
            name = f"{name}_{i:02d}"
        used_names.add(name)

        domains[name] = {
            "lon_min": min(lon_min, lon_max),
            "lon_max": max(lon_min, lon_max),
            "lat_min": min(lat_min, lat_max),
            "lat_max": max(lat_min, lat_max),
        }

    return domains


def subset_box(da: xr.DataArray, box: dict) -> xr.DataArray:
    lat_min, lat_max = sorted([box["lat_min"], box["lat_max"]])
    lon_min, lon_max = sorted([box["lon_min"], box["lon_max"]])

    lat0 = float(da["lat"].values[0])
    lat1 = float(da["lat"].values[-1])
    lon0 = float(da["lon"].values[0])
    lon1 = float(da["lon"].values[-1])

    lat_slice = slice(lat_min, lat_max) if lat0 <= lat1 else slice(lat_max, lat_min)
    lon_slice = slice(lon_min, lon_max) if lon0 <= lon1 else slice(lon_max, lon_min)
    return da.sel(lat=lat_slice, lon=lon_slice)


def union_box_from_domains(domains: OrderedDict[str, dict]) -> dict:
    return {
        "lon_min": min(v["lon_min"] for v in domains.values()),
        "lon_max": max(v["lon_max"] for v in domains.values()),
        "lat_min": min(v["lat_min"] for v in domains.values()),
        "lat_max": max(v["lat_max"] for v in domains.values()),
    }


def compute_region_monthly_mswep(da: xr.DataArray, box: dict) -> pd.Series:
    sub = subset_box(da, box)
    if sub.sizes.get("lat", 0) == 0 or sub.sizes.get("lon", 0) == 0:
        return pd.Series(dtype=float, name="rain_mm_month")

    ser = sub.mean(dim=("lat", "lon"), skipna=True).to_series()
    ser.index = pd.to_datetime(ser.index)
    ser = ser.sort_index().groupby(ser.index.to_period("M")).mean()
    ser.index = ser.index.to_timestamp(how="start")
    ser.name = "rain_mm_month"
    return ser


def build_djf(monthly_series: pd.Series, start_year: int, end_year: int, require_complete: bool = True) -> pd.Series:
    if monthly_series.empty:
        idx = pd.Index(np.arange(start_year, end_year + 1), name="year")
        return pd.Series(index=idx, dtype=float, name="djf_mean")

    s = monthly_series.copy()
    s.index = pd.to_datetime(s.index)
    out = {}
    for y in range(start_year, end_year + 1):
        months = [pd.Timestamp(y - 1, 12, 1), pd.Timestamp(y, 1, 1), pd.Timestamp(y, 2, 1)]
        vals = np.array([s.get(m, np.nan) for m in months], dtype=float)
        if require_complete and not np.all(np.isfinite(vals)):
            out[y] = np.nan
        else:
            out[y] = np.nanmean(vals)

    djf = pd.Series(out, name="djf_mean")
    djf.index.name = "year"
    return djf


def load_nino34_monthly(path: Path) -> pd.Series:
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]

    if "date" not in df.columns:
        raise ValueError(f"Expected 'date' column in {path}. Found: {list(df.columns)}")

    value_cols = [c for c in df.columns if c != "date"]
    if not value_cols:
        raise ValueError(f"No index value column in {path}")

    value_col = value_cols[0]
    tmp = df[["date", value_col]].copy()
    tmp["time"] = pd.to_datetime(tmp["date"], errors="coerce")
    tmp["nino34"] = pd.to_numeric(tmp[value_col], errors="coerce")
    tmp["nino34"] = tmp["nino34"].replace([-9999, -9999.0, -99.99], np.nan)
    tmp = tmp.dropna(subset=["time"])

    tmp["time"] = tmp["time"].dt.to_period("M").dt.to_timestamp(how="start")
    tmp = tmp.groupby("time", as_index=False)["nino34"].mean().sort_values("time")

    s = tmp.set_index("time")["nino34"]
    s.name = "nino34"
    return s


def running_corr(x: pd.Series, y: pd.Series, window: int = 15) -> pd.Series:
    if window % 2 == 0:
        raise ValueError("Window must be odd")

    x = x.sort_index()
    y = y.sort_index()
    years = sorted(set(x.index).intersection(set(y.index)))
    if not years:
        return pd.Series(dtype=float, name="corr")

    years = pd.Index(years, name="year")
    half = window // 2
    rows = []

    for center in range(int(years.min()) + half, int(years.max()) - half + 1):
        ys = pd.Index(np.arange(center - half, center + half + 1), name="year")
        pair = pd.concat([x.reindex(ys).rename("x"), y.reindex(ys).rename("y")], axis=1).dropna()
        if len(pair) != window:
            continue
        rows.append((center, pair["x"].corr(pair["y"])))

    out = pd.Series(dict(rows), name="corr")
    out.index.name = "year_center"
    return out


## Config and Data Load

Load domains, MSWEP monthly data, and monthly Nino3.4.


In [ ]:
START_YEAR = 1980
END_YEAR = 2024
WINDOW = 15

PREFERRED_DOMAIN_ORDER = [
    "Riau",
    "Aceh",
    "Kalimantan",
    "West Java",
    "East Java",
    "Central Sulawesi",
    "West Papua",
    "North Papua",
    "South Papua",
]


def list_mswep_monthly_files(mswep_dir: Path, start_yyyymm: int, end_yyyymm: int) -> list[Path]:
    files = []
    for fp in sorted(mswep_dir.glob("*.nc")):
        stem = fp.stem
        if len(stem) != 6 or not stem.isdigit():
            continue
        yyyymm = int(stem)
        if start_yyyymm <= yyyymm <= end_yyyymm:
            files.append(fp)
    return files


def standardize_mswep_da(da: xr.DataArray) -> xr.DataArray:
    rename_map = {}
    if "latitude" in da.coords or "latitude" in da.dims:
        rename_map["latitude"] = "lat"
    if "longitude" in da.coords or "longitude" in da.dims:
        rename_map["longitude"] = "lon"
    if rename_map:
        da = da.rename(rename_map)

    da = da.assign_coords(time=pd.to_datetime(da["time"].values)).sortby("time")
    da = da.assign_coords(lon=((da["lon"] + 360) % 360)).sortby("lon")

    t0 = pd.Timestamp(START_YEAR - 1, 12, 1)
    t1 = pd.Timestamp(END_YEAR, 2, 28)
    da = da.sel(time=slice(t0, t1))
    return da


domains = load_domains(DOMAIN_JSON)
domain_order = [d for d in PREFERRED_DOMAIN_ORDER if d in domains] + [d for d in domains if d not in PREFERRED_DOMAIN_ORDER]
domains = OrderedDict((k, domains[k]) for k in domain_order)

mswep_files = list_mswep_monthly_files(
    DATA_MSWEP_DIR,
    start_yyyymm=(START_YEAR - 1) * 100 + 12,
    end_yyyymm=END_YEAR * 100 + 2,
)

ds_mswep = xr.open_mfdataset([str(f) for f in mswep_files], combine="by_coords", decode_times=True, parallel=False)
mswep_da = standardize_mswep_da(ds_mswep["precipitation"])

nino_path = DATA_NINO34 if DATA_NINO34.exists() else DATA_NINA34
nino34_monthly = load_nino34_monthly(nino_path)

print("Regions:", domain_order)
print("MSWEP files:", len(mswep_files))
print("Nino source:", nino_path)


## Compute DJF Series and Running Correlation

For each region:
1. Build DJF rainfall series.
2. Build DJF Nino3.4 series.
3. Compute centered running correlation (window = 15).
4. Save tidy tables.


In [ ]:
# Build DJF rainfall + Nino3.4 for common years.
region_series = []
running_records = []

years = pd.Index(np.arange(START_YEAR, END_YEAR + 1), name="year")
nino34_djf = build_djf(nino34_monthly, START_YEAR, END_YEAR, require_complete=True).reindex(years)

for region in domain_order:
    monthly = compute_region_monthly_mswep(mswep_da, domains[region])
    rain_djf = build_djf(monthly, START_YEAR, END_YEAR, require_complete=True).reindex(years)

    for y in years:
        region_series.append(
            {
                "region": region,
                "year": int(y),
                "rain_djf_mm": float(rain_djf.loc[y]) if np.isfinite(rain_djf.loc[y]) else np.nan,
                "nino34_djf": float(nino34_djf.loc[y]) if np.isfinite(nino34_djf.loc[y]) else np.nan,
            }
        )

    rc = running_corr(rain_djf, nino34_djf, window=WINDOW)
    for yc, rv in rc.items():
        running_records.append(
            {
                "region": region,
                "year_center": int(yc),
                "window": WINDOW,
                "corr": float(rv),
            }
        )

# Add a union-box series for all Indonesia.
all_box = union_box_from_domains(domains)
all_monthly = compute_region_monthly_mswep(mswep_da, all_box)
all_rain_djf = build_djf(all_monthly, START_YEAR, END_YEAR, require_complete=True).reindex(years)

for y in years:
    region_series.append(
        {
            "region": "All Indonesia",
            "year": int(y),
            "rain_djf_mm": float(all_rain_djf.loc[y]) if np.isfinite(all_rain_djf.loc[y]) else np.nan,
            "nino34_djf": float(nino34_djf.loc[y]) if np.isfinite(nino34_djf.loc[y]) else np.nan,
        }
    )

all_rc = running_corr(all_rain_djf, nino34_djf, window=WINDOW)
for yc, rv in all_rc.items():
    running_records.append(
        {
            "region": "All Indonesia",
            "year_center": int(yc),
            "window": WINDOW,
            "corr": float(rv),
        }
    )

series_df = pd.DataFrame(region_series).sort_values(["region", "year"])
running_df = pd.DataFrame(running_records).sort_values(["region", "year_center"])

series_csv = OUT_TABLES / f"djf_rain_nino34_series_{START_YEAR}_{END_YEAR}.csv"
run_csv = OUT_TABLES / f"runningcorr_djf_rain_vs_nino34_window{WINDOW}_{START_YEAR}_{END_YEAR}.csv"
series_df.to_csv(series_csv, index=False)
running_df.to_csv(run_csv, index=False)

print("Saved:", series_csv)
print("Saved:", run_csv)
print(running_df.head())


## Plot: Running Correlation by Region

One panel per region. This is the only diagnostic in v3.


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 10), sharex=True, sharey=True)
axes = axes.ravel()

for i, region in enumerate(domain_order):
    ax = axes[i]
    dsub = running_df[running_df["region"] == region]

    ax.plot(dsub["year_center"], dsub["corr"], color="#2166ac", lw=2.0)
    ax.axhline(0, color="0.35", ls="--", lw=0.9)
    ax.set_ylim(-1.0, 1.0)
    ax.set_title(region)

for ax in axes[len(domain_order):]:
    ax.axis("off")

fig.suptitle(f"V3 Running Correlation: DJF Rainfall vs DJF Nino3.4 (Window={WINDOW})")
fig.supxlabel("Center DJF year")
fig.supylabel("Pearson r")
fig.tight_layout(rect=[0, 0, 1, 0.95])
save_and_show(fig, f"v3_runningcorr_djf_rain_vs_nino34_window{WINDOW}.png")


## Plot: Running Correlation for All Indonesia

Single-panel running correlation for the **All Indonesia** union region.


In [ ]:
all_df = running_df[running_df["region"] == "All Indonesia"].copy()

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(all_df["year_center"], all_df["corr"], color="#b2182b", lw=2.2)
ax.axhline(0, color="0.35", ls="--", lw=0.9)
ax.set_ylim(-1.0, 1.0)
ax.set_title(f"Running Correlation: All Indonesia vs DJF Nino3.4 (Window={WINDOW})")
ax.set_xlabel("Center DJF year")
ax.set_ylabel("Pearson r")
fig.tight_layout()
save_and_show(fig, f"v3_runningcorr_all_indonesia_window{WINDOW}.png")


## Output Inventory


In [ ]:
print("Tables:")
for p in sorted(OUT_TABLES.glob("*.csv")):
    print(" -", p.name)

print("\nPNG:")
for p in sorted(OUT_PNG.glob("*.png")):
    print(" -", p.name)
